# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [1]:
%pip install -q smolagents[transformers] wikipedia

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-huggingface 0.1.2 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 0.1.23 which is incompatible.
langchain-huggingface 0.1.2 requires tokenizers>=0.19.1, but you have tokenizers 0.15.2 which is incompatible.
langchain-huggingface 0.1.2 requires transformers>=4.39.0, but you have transformers 4.37.2 which is incompatible.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.


## 1) Define KB

In [1]:
#To-Do: you can add your own knowledge base snippets here
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


KB entries: 5


## 2) Define tools

In [2]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    name = "kb_lookup_tool"
    description = (
        "Looks up relevant information from a custom knowledge base about agentic AI. "
        "Returns matching snippets prefixed with their source id, or 'No KB match.'"
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "The search query or keywords to look up in the knowledge base.",
        }
    }
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {"type": "number", "description": "The first number."},
        "b": {"type": "number", "description": "The second number."},
        "op": {
            "type": "string",
            "description": "The operation to perform: 'add' or 'multiply'.",
            "nullable": True,
        },
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

c:\Users\avigd\Desktop\GENAI\.venv2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3) Model (tiny local)

In [3]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    model_id=MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    max_new_tokens=256,
)

print("Model ready:", MODEL_ID)

Loading weights: 100%|██████████| 201/201 [00:01<00:00, 123.12it/s]


Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [4]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        "You are an agentic AI assistant that must use tools to answer questions. "
        "Use `kb_lookup_tool` for any conceptual or factual question about agentic AI, "
        "and `math_tool` for any arithmetic (addition or multiplication). "
        "Always cite the source id returned by the knowledge base, e.g. [kb:agentic]. "
        "If the knowledge base returns 'No KB match.', say that no evidence was found "
        "and propose a follow-up question instead of inventing an answer. "
        "Keep your final answer to 2-4 sentences."
    ),
)

print(agent)

## 5) Test queries

In [6]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent.run(q)
    print("Answer:", result)

---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - TinyLlama/TinyLlama-1.1B-Chat-v1.0 ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'addition_tool' with arguments: {'num1': '12', 'num2': '30'}                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Unknown tool addition_tool, should be one of: kb_lookup_tool, math_tool, final_answer.

... (34 outputs left out)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 3: Duration 2.44 seconds| Input tokens: 3,305 | Output tokens: 364]

Answer: An agentic AI loop is a series of actions that repeat indefinitely, with each action being a tool call. The loop is typically used to solve complex problems or answer questions that require a specific pattern or sequence of actions. The actions in an agentic AI loop are typically designed to perform a specific task or function, and the loop repeats until the task is completed or until a specific condition is met.
